In [1]:
import socket
import json

### Step 1: TCP connection

In [2]:
def stream_records(host='localhost', port=9034):
    sock = socket.create_connection((host, port))
    buffer = ""

    try:
        while True:
            buffer += sock.recv(4096).decode()
            while '\n' in buffer:
                line, buffer = buffer.split('\n', 1)
                line = line.strip()
                if not line:
                    continue
                try:
                    yield json.loads(line)
                except json.JSONDecodeError:
                    yield {'_error': 'invalid_json', 'raw':line}
    finally:
        sock.close()

### Step 2: validation

In [3]:
from datetime import datetime

VALID_STATIONS = {f'ST-{i:02d}' for i in range(1, 11)}
MODELS = {'leak_detector', 'pressure_drop_predictor', 'demand_forecaster'}

def validate(rec):
    errors = []
    rid = rec.get('reading_id')
    if not isinstance(rid, int) or rid <= 0:
        errors.append('reading_id must be positive int')
    if rec.get('station_id') not in VALID_STATIONS:
        errors.append('invalid station_id')

    model = rec.get('model_name')
    if model not in MODELS:
        errors.append('invalid model_name')
    else:
        label = rec.get('predicted_label')
        if model == "demand_forecaster":
            if not isinstance(label, int) or not (1 <= label <= 5000):
                errors.append("predicted_label must be int 1..5000")
        else:
            allowed = {"leak_detector": {"leak", "normal"},
                       "pressure_drop_predictor": {"drop", "stable"}}
            if label not in allowed[model]:
                errors.append("predicted_label mismatch for model")


    conf = rec.get("confidence_score")
    if not isinstance(conf, (int, float)) or not (0 <= conf <= 1):
        errors.append("confidence_score must be in [0,1]")

    rt = rec.get("response_time_ms")
    if not isinstance(rt, (int, float)) or rt <= 0:
        errors.append("response_time_ms must be positive")

    ts = rec.get("timestamp")
    try:
        datetime.fromisoformat(str(ts))
    except (ValueError, TypeError):
        errors.append("invalid timestamp")

    return errors

### Step 3: report and saving

In [4]:
import csv, time

def main():
    clean_f = open("clean_readings.csv", "a", newline="")
    bad_f   = open("bad_readings.csv", "a", newline="")
    rep_f   = open("real_time_reports.csv", "a", newline="")

    clean_w = csv.DictWriter(clean_f, fieldnames=[
        "reading_id","station_id","model_name","predicted_label",
        "confidence_score","response_time_ms","timestamp"])
    bad_w   = csv.DictWriter(bad_f, fieldnames=[
        "reading_id","station_id","model_name","predicted_label",
        "confidence_score","response_time_ms","timestamp","errors"], extrasaction='ignore')
    rep_w   = csv.DictWriter(rep_f, fieldnames=[
        "total","clean","bad","active_stations"])

    clean_w.writeheader(); bad_w.writeheader(); rep_w.writeheader()

    total = clean = bad = 0
    active = set()
    last_report = time.time()

    for rec in stream_records():
        total += 1
        if "_error" in rec:
            bad += 1
            bad_w.writerow({**rec, "errors": rec["_error"]})
        else:
            errs = validate(rec)
            if errs:
                bad += 1
                row = {**rec, "errors": "; ".join(errs)}
                bad_w.writerow(row)
                bad_f.flush()
            else:
                clean += 1
                active.add(rec["station_id"])
                clean_w.writerow(rec)

        if time.time() - last_report >= 20:
            rep_w.writerow({"total": total, "clean": clean,
                            "bad": bad, "active_stations": len(active)})
            rep_f.flush()
            total = clean = bad = 0
            active.clear()
            last_report = time.time()

In [6]:
if __name__ == "__main__":
    main()

KeyboardInterrupt: 